## tl;dr

The duplicate-group evaluation removed 117 false positives and 795 true positives with symbolic filtering. The development-selected score filter removed 110 and 11, respectively. This supports a substantial recall-cost caveat, not operational superiority. Capture-day challenges are reported separately below.

## Context & Methods

This is a lightweight companion to the dated corrective protocol, using published aggregate results only. It checks reported arithmetic and evidence hashes, not a new forest fit. The full reproduction command and input hashes are in the repository README and experiment manifest.

### Key Assumptions

All removal counts use the same 0.15 reference queue within each condition. Splits are exact-feature-group separated with training-only imputation. Thresholds were selected on development records. Days contain different attacks, Monday has no malicious examples, and the five day-held-out models must not be pooled as independent observations. Historical dataset results were already known; this is not external confirmation.

## Data

Load only the published aggregate files. No credentials, network request or raw traffic file is needed.

In [1]:
import json, hashlib
from pathlib import Path
root = Path.cwd()
if not (root / 'evidence').exists(): root = root.parent
evidence = root / 'evidence' / '20260905'
manifest = json.loads((evidence / 'manifest.json').read_text())
results = json.loads((evidence / 'results.json').read_text())
checks = json.loads((evidence / 'corrective_verification.json').read_text())
assert manifest['status'] == 'complete' and checks['status'] == 'passed'
assert len(results) == 6
print('Six completed conditions; independent local reconciliation passed.')

Six completed conditions; independent local reconciliation passed.


### Validate published evidence and source identities

In [2]:
for relative, expected in manifest['source_hashes'].items():
    path = root / relative.replace('\\', '/')
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, relative
for relative, expected in manifest['output_hashes'].items():
    path = evidence / relative.replace('\\', '/')
    if path.exists():
        assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, relative
for result in results:
    assert not any(result['split_checks']['overlap_checks'].values())
    assert result['smt_disagreements'] == 0
print('Available original evidence hashes, source hashes, overlap and SMT checks passed.')

Available original evidence hashes, source hashes, overlap and SMT checks passed.


## Results

Rates below are recomputed from integer counts, including undefined recall where a test day contains no malicious examples.

In [3]:
for result in results:
    for policy, m in result['policies'].items():
        assert m['baseline_tp'] - m['retained_tp'] == m['tp_removed']
        assert m['baseline_fp'] - m['retained_fp'] == m['fp_removed']
        assert m['validated_fn'] == m['baseline_fn'] + m['tp_removed']
        expected = -m['tp_removed'] / m['malicious'] if m['malicious'] else None
        assert m['recall_delta'] == expected
    s = result['policies']['symbolic']
    recall = 'undefined' if s['validated_recall'] is None else f"{100*s['validated_recall']:.3f}%"
    print(result['condition'], '| FP/TP removed:', s['fp_removed'], s['tp_removed'], '| retained recall:', recall)

duplicate_group_split | FP/TP removed: 117 795 | retained recall: 96.467%
leave_day_out_Monday | FP/TP removed: 153 0 | retained recall: undefined
leave_day_out_Tuesday | FP/TP removed: 290 303 | retained recall: 19.536%
leave_day_out_Wednesday | FP/TP removed: 29 3213 | retained recall: 6.983%
leave_day_out_Thursday | FP/TP removed: 10491 1835 | retained recall: 1.489%
leave_day_out_Friday | FP/TP removed: 222 87 | retained recall: 30.956%


### Compare the locked group-split policies

In [4]:
main = results[0]
assert main['selection']['selection_partition'] == 'development_only'
for name, m in main['policies'].items():
    print(name, '| FP removed:', m['fp_removed'], '| TP removed:', m['tp_removed'], '| recall change (percentage points):', round(100*m['recall_delta'], 4))
print('Selected score cutoff:', main['selection']['score_only_cutoff'])
print('Selected detector cutoff:', main['selection']['detector_cutoff'])

development_detector | FP removed: 17 | TP removed: 1 | recall change (percentage points): -0.0042
development_detector_plus_symbolic | FP removed: 121 | TP removed: 796 | recall change (percentage points): -3.3725
development_score_filter | FP removed: 110 | TP removed: 11 | recall change (percentage points): -0.0466
reference_detector | FP removed: 0 | TP removed: 0 | recall change (percentage points): 0.0
symbolic | FP removed: 117 | TP removed: 795 | recall change (percentage points): -3.3682
Selected score cutoff: 0.21
Selected detector cutoff: 0.16


### Inspect the practical-cost boundary

This is an assumption grid, not a measurement of analyst time or incident cost.

In [5]:
s = main['policies']['symbolic']
for relative_cost in [1, 5, 10, 20, 100, 1000]:
    benefit = s['fp_removed'] - relative_cost*s['tp_removed']
    print('Assumed FN:FP cost', relative_cost, '| net count benefit', benefit)
print('Break-even FN:FP cost:', s['fp_removed']/s['tp_removed'])

Assumed FN:FP cost 1 | net count benefit -678
Assumed FN:FP cost 5 | net count benefit -3858
Assumed FN:FP cost 10 | net count benefit -7833
Assumed FN:FP cost 20 | net count benefit -15783
Assumed FN:FP cost 100 | net count benefit -79383
Assumed FN:FP cost 1000 | net count benefit -794883
Break-even FN:FP cost: 0.1471698113207547


## Takeaways

The locked comparator preserves substantially more malicious alerts, while removing seven fewer benign alerts in the group split; this is not strict dominance on both counts. A point estimate meeting a historical target does not make that target operationally safe. The capture-day results expose transfer limits, and near-related flows and benchmark label defects remain uncontrolled. Latency is serial local enrichment plus validation only. Full model reproduction requires independently obtained matching CICIDS2017 inputs and the pinned software environment.